# Flink SQL Exploration

This notebook demonstrates querying the streaming lakehouse using Flink SQL via the SQL Gateway.

## Prerequisites

1. Start the stack: `make up`
2. Seed infrastructure: `make seed`
3. Run a streaming job to populate Iceberg tables

## What You'll Learn

- Connect to Flink SQL Gateway programmatically
- Query Iceberg tables with SQL
- Perform time-travel queries
- Explore table metadata and snapshots
- Analyze streaming data with window functions

## Setup

In [ ]:
# Import required libraries
import os
import json
import requests
import pandas as pd
from typing import Any

# Flink SQL Gateway endpoint
SQL_GATEWAY_URL = os.getenv("FLINK_SQL_GATEWAY_URL", "http://localhost:8083")
print(f"Connecting to Flink SQL Gateway: {SQL_GATEWAY_URL}")

## SQL Gateway Client

Create a simple client to execute SQL statements via the REST API.

In [ ]:
class FlinkSQLClient:
    """Simple client for Flink SQL Gateway REST API."""
    
    def __init__(self, gateway_url: str):
        self.gateway_url = gateway_url
        self.session_handle = None
    
    def create_session(self) -> str:
        """Create a new SQL session."""
        response = requests.post(f"{self.gateway_url}/v1/sessions")
        response.raise_for_status()
        self.session_handle = response.json()["sessionHandle"]
        print(f"Created session: {self.session_handle}")
        return self.session_handle
    
    def execute_sql(self, sql: str) -> pd.DataFrame:
        """Execute SQL and return results as DataFrame."""
        if not self.session_handle:
            self.create_session()
        
        # Submit SQL statement
        payload = {"statement": sql}
        response = requests.post(
            f"{self.gateway_url}/v1/sessions/{self.session_handle}/statements",
            json=payload
        )
        response.raise_for_status()
        operation_handle = response.json()["operationHandle"]
        
        # Fetch results (simplified - production should poll for completion)
        result_response = requests.get(
            f"{self.gateway_url}/v1/sessions/{self.session_handle}/operations/{operation_handle}/result/0"
        )
        result_response.raise_for_status()
        result = result_response.json()
        
        # Convert to DataFrame
        if "results" in result and "data" in result["results"]:
            columns = [col["name"] for col in result["results"]["columns"]]
            data = result["results"]["data"]
            return pd.DataFrame(data, columns=columns)
        return pd.DataFrame()
    
    def close_session(self):
        """Close the SQL session."""
        if self.session_handle:
            requests.delete(f"{self.gateway_url}/v1/sessions/{self.session_handle}")
            print(f"Closed session: {self.session_handle}")

# Create client instance
sql_client = FlinkSQLClient(SQL_GATEWAY_URL)

## Query Iceberg Catalog

List available databases and tables in the Iceberg catalog.

In [ ]:
# List catalogs
catalogs_df = sql_client.execute_sql("SHOW CATALOGS")
display(catalogs_df)

In [ ]:
# Use Iceberg catalog
sql_client.execute_sql("USE CATALOG iceberg_catalog")

# List databases
databases_df = sql_client.execute_sql("SHOW DATABASES")
display(databases_df)

In [ ]:
# List tables in streaming_lakehouse database
sql_client.execute_sql("USE streaming_lakehouse")
tables_df = sql_client.execute_sql("SHOW TABLES")
display(tables_df)

## Query Iceberg Tables

Execute analytical queries on the Iceberg tables.

In [ ]:
# Example: Query enriched ticks (replace with your actual table name)
query = """
SELECT 
    symbol,
    COUNT(*) as tick_count,
    AVG(price) as avg_price,
    MIN(price) as min_price,
    MAX(price) as max_price
FROM ticks_enriched
GROUP BY symbol
ORDER BY tick_count DESC
LIMIT 10
"""

result_df = sql_client.execute_sql(query)
display(result_df)

## Time-Travel Queries

Query historical snapshots of Iceberg tables.

In [ ]:
# Query table as of specific timestamp
# TODO: Replace with actual timestamp and table name
time_travel_query = """
SELECT * FROM ticks_enriched 
FOR SYSTEM_TIME AS OF TIMESTAMP '2024-01-01 12:00:00'
LIMIT 10
"""

# result_df = sql_client.execute_sql(time_travel_query)
# display(result_df)
print("Time-travel query example (update timestamp and uncomment to execute)")

## Cleanup

In [ ]:
# Close SQL session
sql_client.close_session()

## Next Steps

- Modify queries to match your actual table schemas
- Explore window functions for time-series analysis
- Join streaming and batch tables
- Experiment with Iceberg table maintenance (compaction, expire snapshots)
- See `notebooks/iceberg_inspect.ipynb` for low-level Iceberg metadata exploration